# DumpWatch 2.0 · Task B exploratory SVM comparison

Predict whether the deposited waste is fully cleared or whether some waste remains. The two arms use identical real training pairs, preprocessing, 723-element pair features, fixed linear SVM settings, and a development-only comparison. Arm B adds V1 appearance supervision. The reserved real test image members are never opened.

The reviewed pair CSV is the experiment configuration. Six ROI pairs were widened during review to keep visible deposited waste within the crop. Source filenames, labels, and membership are unchanged. Comparable ground rectangles substitute for geometric registration in this small experiment. The score rules are fixed before inspecting development outputs. This notebook executes the approved exploratory experiment on CPU.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
INPUTS = P["manifest_root"]
REAL_ZIP = P["real_zip"]
SPLIT_CSV = INPUTS / "real_location_split.csv"
PAIR_CSV = INPUTS / "task_b_exploratory_pairs.csv"
ALLOWLIST = INPUTS / "synthetic_v1_allowlist.json"
SYNTHETIC_ROOT = P["synthetic_root"]
V1_LAYOUT = P["synthetic_layout"]
OUTPUT_ROOT = new_output("real_vs_v1_training")
print("Historical development experiment. Following cells TRAIN new experimental models into:", OUTPUT_ROOT)


## 1. Verify source data and prepare features

Real descriptors use 4 × 4 spatial cells over a 128 × 128 resized ground crop. Each cell contributes nine normalized unsigned gradient bins and six RGB mean and standard-deviation values. Pair features concatenate W, inspection, absolute feature differences, and three appearance-score terms. The V1 selection uses one allowlisted W crop and its same-region B0 crop per location for waste appearance only. The V1 copied inspection branches never supervise the paired SVM.


In [ ]:
import csv, hashlib, io, json, zipfile
from collections import Counter, defaultdict
import numpy as np
from PIL import Image

TRAIN = {f"location_{n:02}" for n in (1,3,5,6,7,9)}
DEV = {"location_02", "location_08"}
TEST = {"location_04", "location_10"}
REAL_HASH = "1e97cfe51b4ef7fc9e61aaa7cbd2cbce4ac5a24fd274f07503d7deb5138f0412"
SPLIT_HASH = "0ae3ff815747ba5c8c677fb3670c373AECBE9ECF2B4E3BD6B2410450748A3E60".lower()

def sha(data):
    return hashlib.sha256(data).hexdigest()

def require(ok, message):
    if not ok:
        raise ValueError(message)

def box_from_string(value):
    return [float(n) for n in value.split()]

def descriptor(raw, box):
    left,top,right,bottom=box
    require(0<=left<right<=1 and 0<=top<bottom<=1, f"Invalid normalized ROI {box}")
    with Image.open(io.BytesIO(raw)) as source:
        w,h=source.size
        pixels=source.convert("RGB").crop((round(left*w),round(top*h),round(right*w),round(bottom*h)))
    require(min(pixels.size)>=64, "ROI too small to assess")
    rgb=np.asarray(pixels.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255
    gray=rgb @ np.array([.299,.587,.114],dtype=np.float32)
    dy,dx=np.gradient(gray)
    magnitude=np.hypot(dx,dy)
    orientation=np.floor((np.arctan2(dy,dx)%np.pi)*9/np.pi).astype(int).clip(0,8)
    features=[]
    for y in range(0,128,32):
        for x in range(0,128,32):
            region=np.s_[y:y+32,x:x+32]
            hist=np.bincount(orientation[region].ravel(),weights=magnitude[region].ravel(),minlength=9)
            features.extend(hist/(np.linalg.norm(hist)+1e-6))
            patch=rgb[region]
            features.extend(patch.mean(axis=(0,1)))
            features.extend(patch.std(axis=(0,1)))
    return np.asarray(features,dtype=np.float32)

def paired_vector(w,a,scores):
    return np.r_[w,a,np.abs(w-a),scores[0],scores[1],scores[0]-scores[1]]

def select_clearance(score, visually_assessable, low, high):
    require(np.isfinite(low) and np.isfinite(high) and low<high,"Invalid abstention interval")
    return np.where(~np.asarray(visually_assessable,dtype=bool),"abstain",
           np.where(score>=high,"fully_cleared",np.where(score<=low,"waste_still_present","abstain")))

def evaluate(target,prediction):
    target,prediction=np.asarray(target),np.asarray(prediction)
    remaining=target==0
    cleared=target==1
    false=int((remaining & (prediction=="fully_cleared")).sum())
    valid=int((prediction!="abstain").sum())
    return dict(false_clearance=false,remaining_total=int(remaining.sum()),
        false_clearance_rate=false/remaining.sum() if remaining.any() else None,
        correctly_cleared=int((cleared & (prediction=="fully_cleared")).sum()),
        cleared_total=int(cleared.sum()),cleared_as_waste_present=int((cleared & (prediction=="waste_still_present")).sum()),
        waste_present_correct=int((remaining & (prediction=="waste_still_present")).sum()),
        evaluated=len(target),coverage=valid/len(target) if len(target) else None,
        abstentions=int(len(target)-valid))

In [ ]:
def real_features():
    require(sha(REAL_ZIP.read_bytes())==REAL_HASH,"Real ZIP is not the frozen archive")
    require(sha(SPLIT_CSV.read_bytes())==SPLIT_HASH,"Frozen split CSV mismatch")
    with SPLIT_CSV.open(newline="",encoding="utf-8-sig") as f:
        mapping={x["location_id"]:x["split"] for x in csv.DictReader(f)}
    require(mapping=={**dict.fromkeys(TRAIN,"train"),**dict.fromkeys(DEV,"validation"),**dict.fromkeys(TEST,"test")},
            "Unexpected split membership")
    with PAIR_CSV.open(newline="",encoding="utf-8-sig") as f:
        review=list(csv.DictReader(f))
    require(len(review)==48 and len({r["pair_id"] for r in review})==48,"Unexpected pair inventory")
    require({r["location_id"] for r in review}==TRAIN|DEV,"Unexpected or reserved pair location")
    selected=[r for r in review if r["status"]=="usable"]
    seen=set()
    result=[]
    with zipfile.ZipFile(REAL_ZIP) as z:
        archived={r["relative_path"]:r for r in csv.DictReader(io.StringIO(z.read("dataset_manifest.csv").decode("utf-8-sig")))}
        for row in selected:
            loc=row["location_id"]
            split="training" if loc in TRAIN else "development"
            require(row["split"]==split,f"Incorrect split for {row['pair_id']}")
            require(row["inspection_sha256"] not in seen,"Duplicate eligible inspection hash")
            seen.add(row["inspection_sha256"])
            role=row["inspection_stage"]
            require(role in {"A_partial","A_removed"},"Dependent unchanged controls are excluded")
            target=int(role=="A_removed")
            require(row["target"]==("fully_cleared" if target else "waste_still_present"),"Target mismatch")
            vectors=[]
            for column,hash_column,roi_column,stage in (
                    ("W_path","W_sha256","W_roi_xyxy","W"),
                    ("inspection_path","inspection_sha256","inspection_roi_xyxy",role)):
                path=row[column]
                frozen=archived[path]
                # Enforce allowed location and same case before opening any member.
                require(path.startswith(f"real/{loc}/") and frozen["location_id"]==loc and loc not in TEST
                        and frozen["case_id"]==row["case_id"] and frozen["semantic_stage"]==stage,
                        f"Unexpected member: {path}")
                raw=z.read(path)
                require(sha(raw)==row[hash_column]==frozen["file_hash"],f"Image hash mismatch: {path}")
                vectors.append(descriptor(raw,box_from_string(row[roi_column])))
            require(row["W_sha256"]!=row["inspection_sha256"],"Exact W control in real index")
            result.append(dict(row,w=vectors[0],a=vectors[1],y=target))
    hash_membership=defaultdict(set)
    for row in result:
        for field in ("W_sha256","inspection_sha256"):
            hash_membership[row[field]].add(row["split"])
    require(all(len(v)==1 for v in hash_membership.values()),"A real image hash crosses splits")
    return result,review

def v1_appearance():
    plan=json.loads(ALLOWLIST.read_text(encoding="utf-8"))
    expected={(f"location {n}", f"case_{k:03}") for n in range(1,12) for k in range(1,9)}
    require(len(plan["cases"])==88 and {(r["location"],r["case"]) for r in plan["cases"]}==expected
            and plan["synthetic_only"] is True,"Unexpected V1 allowlist")
    sample=[]
    seen=set()
    def verified(item, expected_path):
        require(item["path"]==expected_path,"V1 item is outside expected exact path")
        parts=expected_path.split("/")
        if V1_LAYOUT=="archive_mirror":
            parts[0]=f"location_{int(parts[0].split()[-1]):02}"
        else:
            require(V1_LAYOUT=="drive","Unknown V1 layout")
        raw=SYNTHETIC_ROOT.joinpath(*parts).read_bytes()
        require(len(raw)==item["size"] and sha(raw)==item["sha256"],f"V1 hash mismatch: {expected_path}")
        return raw
    for row in plan["cases"]:
        if row["case"]!="case_008":
            continue  # one representative deposit per V1 location
        loc=row["location"]
        require(row["split"]=="synthetic_train_only","Unexpected V1 role")
        path=f"{loc}/scenarios/"
        meta=json.loads(verified(row["metadata"],path+"case_008/metadata.json"))
        roi=np.asarray(meta["waste_placement_region"]["normalized_mask_bbox"],float)
        span=roi[2:]-roi[:2]
        box=np.clip(np.r_[roi[:2]-.22*span,roi[2:]+.22*span],0,1)
        for item,file,y in ((row["images"]["W"],"case_008/W_deposited.png",1),
                            (plan["baselines"][loc],"B0_reference.png",0)):
            if item["sha256"] in seen:
                continue
            seen.add(item["sha256"])
            f=descriptor(verified(item,path+file),box)
            sample.append(dict(x=f,y=y,site=loc,sha256=item["sha256"]))
    require(len(sample)==22 and Counter(r["y"] for r in sample)=={0:11,1:11},
            "Unexpected V1 auxiliary coverage after deduplication")
    return sample

real,review=real_features()
aux=v1_appearance()
require(not {a["sha256"] for a in aux} & {r[k] for r in real for k in ("W_sha256","inspection_sha256")},
        "V1 appearance hash overlaps the real pair set")
print("Real usable pairs:",dict(Counter((r["split"],r["target"]) for r in real)))
print("Held/excluded by status:",dict(Counter(r["status"] for r in review if r["status"]!="usable")))
print("V1 auxiliary appearance crops:",len(aux),"(11 W and 11 B0, training side only)")
print("Prepared descriptors:",real[0]["w"].shape,"; no estimator fitted.")

## 2. Fit comparable models

Each arm fits an appearance SVM and a paired SVM. Out-of-location appearance scores feed the real training pairs. Standardization and SVM fitting use training data only. The fixed hyperparameter is C = 0.1 for every linear model, with balanced class weights. The auxiliary samples receive capped weights. Related outcomes share their case weight. No development scores select the model or cutoff.


In [ ]:
def appearance_records(rows):
    unique={}
    for row in rows:
        for part,feature,y in (("W_sha256",row["w"],1),
                               ("inspection_sha256",row["a"],1-row["y"])):
            key=row[part]
            current=unique.get(key)
            require(current is None or current["y"]==y,"Conflicting appearance labels")
            unique.setdefault(key,dict(x=feature,y=y,site=row["location_id"],sha256=key))
    return list(unique.values())

def make_svm():
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    return make_pipeline(StandardScaler(),LinearSVC(C=.1,class_weight="balanced",random_state=37,max_iter=20000))

def appearance_model(rows, synthetic):
    records=appearance_records(rows)
    train=records+synthetic
    estimator=make_svm()
    # Equal treatment across arms, with capped auxiliary mass in the assisted arm.
    weight=np.r_[np.ones(len(records)),np.full(len(synthetic),min(.25,len(records)/(4*max(1,len(synthetic)))))]
    x=np.stack([r["x"] for r in train]); y=np.array([r["y"] for r in train])
    estimator.fit(x,y,standardscaler__sample_weight=weight,linearsvc__sample_weight=weight)
    return estimator

def vector_for(model,row):
    scores=model.decision_function(np.stack((row["w"],row["a"])))
    return paired_vector(row["w"],row["a"],scores)

def fit_arm(train,development,synthetic):
    sites=sorted({r["location_id"] for r in train})
    weights=Counter((r["location_id"],r["case_id"]) for r in train)
    oof=[]
    for site in sites:
        inner=[r for r in train if r["location_id"]!=site]
        held=[r for r in train if r["location_id"]==site]
        appearance=appearance_model(inner,synthetic)
        oof.extend((vector_for(appearance,r),r["y"],1/weights[r["location_id"],r["case_id"]]) for r in held)
    paired=make_svm()
    x,y,w=zip(*oof)
    paired.fit(np.stack(x),np.asarray(y),standardscaler__sample_weight=w,linearsvc__sample_weight=w)
    appearance=appearance_model(train,synthetic)
    X=np.stack([vector_for(appearance,r) for r in development])
    return paired.decision_function(X)

def development_operating_points(scores, development):
    target=np.array([r["y"] for r in development])
    sites=np.array([r["location_id"] for r in development])
    # Fixed exploratory score grid shared across the arms. No claim of calibration.
    reports=[]
    for low,high in ((-.25,.25),(-.5,.5),(-.5,1.),(-.5,1.5)):
        predictions=select_clearance(scores,np.ones(len(scores),bool),low,high)
        reports.append(dict(lower=low,upper=high,overall=evaluate(target,predictions),
            by_location={s:evaluate(target[sites==s],predictions[sites==s]) for s in sorted(set(sites))}))
    return reports

## 3. Development scores and acceptance rule

Use lower and upper signed SVM margins of −0.5 and +0.5. A score inside the interval is uncertain. A pair without sufficient visual coverage would also abstain upstream. These are uncalibrated margins and the cutoff is a provisional symmetric rule from the preparation grid. The zero-margin comparison shows the effect of abstention and is not used to select a threshold. Here, Fully Cleared is the positive class.


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import platform

require(all(r["roi_status"]=="visually_reviewed_exploratory" for r in real),"Unexpected ROI review status")
train=[r for r in real if r["split"]=="training"]
dev=[r for r in real if r["split"]=="development"]
require(Counter(r["y"] for r in train)=={0:14,1:6} and Counter(r["y"] for r in dev)=={0:6,1:2},
        "Reviewed class counts changed")
for held in TRAIN:
    require({r["y"] for r in train if r["location_id"]!=held}=={0,1},"Appearance fold lacks a target class")

MARGIN=0.5
LABELS=("waste_still_present","fully_cleared","abstain")
TARGETS=LABELS[:2]
arm_configs=(("A_real_only",[]),("B_real_plus_V1",aux))
run_dir=OUTPUT_ROOT/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True,exist_ok=False)

def quotient(n,d):
    return n/d if d else None

def metric_summary(rows, column):
    matrix=[[sum(r["actual"]==a and r[column]==b for r in rows) for b in LABELS] for a in TARGETS]
    tn,fp,an=matrix[0]
    fn,tp,ap=matrix[1]
    accepted=tn+fp+fn+tp
    return dict(n=len(rows),matrix=matrix,row_order=list(TARGETS),column_order=list(LABELS),
        accepted=accepted,abstentions=an+ap,coverage=quotient(accepted,len(rows)),
        accepted_accuracy=quotient(tn+tp,accepted),
        fully_cleared_precision=quotient(tp,tp+fp),
        fully_cleared_recall_accepted=quotient(tp,tp+fn),
        fully_cleared_f1_accepted=quotient(2*tp,2*tp+fp+fn),
        fully_cleared_recall_all=quotient(tp,tp+fn+ap),
        false_clearance_count=fp,false_clearance_denominator=tn+fp+an,
        false_clearance_rate=quotient(fp,tn+fp+an),
        false_alarm_count=fn,false_alarm_denominator=tp+fn+ap,
        false_alarm_rate=quotient(fn,tp+fn+ap),
        abstain_waste_present=an,abstain_fully_cleared=ap)

result=dict(scope="exploratory development only",positive_class="fully_cleared",margin=MARGIN,C=.1,
    implementation=dict(python=platform.python_version(),numpy=version("numpy"),
                        scikit_learn=version("scikit-learn"),pillow=version("Pillow")),
    real_archive_sha256=REAL_HASH,split_sha256=SPLIT_HASH,pair_csv_sha256=sha(PAIR_CSV.read_bytes()),
    v1_allowlist_sha256=sha(ALLOWLIST.read_bytes()),v1_auxiliary_sha256=[r["sha256"] for r in aux],
    real_train_pairs=len(train),real_dev_pairs=len(dev),test_pairs_opened=0,
    development_majority_reference={"correct":6,"total":8},arms={})
predictions=[]
for arm,extra in arm_configs:
    scores=np.asarray(fit_arm(train,dev,extra),dtype=float)
    require(scores.shape==(8,) and np.isfinite(scores).all(),"Invalid development decision scores")
    accepted=select_clearance(scores,np.ones(len(dev),bool),-MARGIN,MARGIN)
    forced=np.where(scores>=0,"fully_cleared","waste_still_present")
    records=[]
    for row,score,pred,decision in zip(dev,scores,accepted,forced):
        records.append(dict(arm=arm,pair_id=row["pair_id"],location_id=row["location_id"],
            case_id=row["case_id"],W_path=row["W_path"],inspection_path=row["inspection_path"],
            actual=TARGETS[row["y"]],score=float(score),prediction=str(pred),forced_prediction=str(decision),
            abstention_reason="signed margin inside (-0.5, 0.5)" if pred=="abstain" else "",
            capture_provenance=row["provenance"]))
    predictions.extend(records)
    result["arms"][arm]=dict(auxiliary_crops=len(extra),
        provisional=metric_summary(records,"prediction"),forced_zero=metric_summary(records,"forced_prediction"),
        by_location={loc:dict(provisional=metric_summary([r for r in records if r["location_id"]==loc],"prediction"),
                              forced_zero=metric_summary([r for r in records if r["location_id"]==loc],"forced_prediction"))
                     for loc in sorted(DEV)})
    with (run_dir/f"predictions_{arm}.csv").open("x",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f,fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
with (run_dir/"predictions_all.csv").open("x",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=list(predictions[0]))
    writer.writeheader()
    writer.writerows(predictions)
(run_dir/"metrics.json").write_text(json.dumps(result,indent=2,allow_nan=False)+"\n",encoding="utf-8")
for arm,m in result["arms"].items():
    print(arm)
    print("  Fixed margin:",m["provisional"])
    print("  Forced zero:",m["forced_zero"])
print("Exploratory result folder:",run_dir)


## 4. Visual review and limits

The report shows both arms on the same eight real development pairs, including false clearance, false alarms, and abstentions. Class precision, recall and F1 are conditioned on accepted predictions, while false clearance uses all actual waste-still-present inspections and coverage is reported separately. Small development counts and two physical sites limit any comparison. A fixed-view synthetic shortcut score is not a real model result.


In [ ]:
from PIL import ImageDraw,ImageOps

INK="#162235"
MUTED="#455873"
canvas=Image.new("RGB",(1380,630),"#f2f5fa")
d=ImageDraw.Draw(canvas)
def text(x,y,label,size=19,fill=INK):
    d.text((x,y),str(label),fill=fill,font_size=size)
text(25,18,"DumpWatch Task B  |  Exploratory real development results",29)
text(25,56,"Eight paired inspections at two locations  |  Fixed abstention interval: -0.5 to +0.5",17,MUTED)
for i,(arm,_) in enumerate(arm_configs):
    x=18+i*680
    d.rounded_rectangle((x,98,x+658,606),radius=15,fill="white",outline="#dce3ef")
    text(x+19,116,("Experiment A  |  Real only","Experiment B  |  Real + V1 appearance")[i],24)
    metrics=result["arms"][arm]["provisional"]
    forced=result["arms"][arm]["forced_zero"]
    for j,title in enumerate(("Present","Cleared","Abstain")):
        text(x+174+j*136,165,title,17,MUTED)
    for y,row in enumerate(metrics["matrix"]):
        text(x+19,213+y*65,("Waste present","Fully cleared")[y],17)
        for j,n in enumerate(row):
            tx,ty=x+170+j*136,197+y*65
            color="#d7eee8" if j==y else "#f2d8cf" if n and j!=2 else "#edf1f7"
            d.rounded_rectangle((tx,ty,tx+122,ty+55),radius=7,fill=color)
            text(tx+50,ty+13,n,23)
    text(x+22,335,f"Coverage: {metrics['accepted']}/8   |   Abstentions: {metrics['abstentions']}/8",18)
    text(x+22,367,f"False clearance: {metrics['false_clearance_count']}/6   |   False alarms: {metrics['false_alarm_count']}/2",18)
    text(x+22,399,f"Forced zero-margin accuracy: {(forced['accepted_accuracy'] or 0)*100:.1f}%",18)
    for j,(loc,val) in enumerate(result["arms"][arm]["by_location"].items()):
        m=val["provisional"]
        text(x+22,453+j*39,f"{loc[-2:]}   Accepted {m['accepted']}/{m['n']}   |   False clearance {m['false_clearance_count']}/{m['false_clearance_denominator']}",17,MUTED)
    text(x+22,551,"Rows show the true class. Columns show predictions.",15,MUTED)
canvas.save(run_dir/"comparison.png")

gallery=Image.new("RGB",(1160,216*len(dev)+54),"#f2f5fa")
g=ImageDraw.Draw(gallery)
g.text((16,12),"All eight real development pairs  |  W at left, inspection at right",fill=INK,font_size=25)
def crop_member(archive,path,region):
    # Gallery uses only the exact paths from the reviewed development pairs.
    require(any(path in (r["W_path"],r["inspection_path"]) for r in dev),"Non-development gallery path")
    with Image.open(io.BytesIO(archive.read(path))) as im:
        w,h=im.size
        l,t,r,b=box_from_string(region)
        return im.convert("RGB").crop((round(l*w),round(t*h),round(r*w),round(b*h)))
with zipfile.ZipFile(REAL_ZIP) as archive:
    for i,row in enumerate(dev):
        y=54+216*i
        g.rounded_rectangle((12,y,1148,y+203),radius=10,fill="white")
        g.text((25,y+12),row["pair_id"],fill=INK,font_size=19)
        g.text((25,y+43),"Truth: "+TARGETS[row["y"]].replace("_"," "),fill=MUTED,font_size=17)
        for j,(key,_) in enumerate(arm_configs):
            pred=next(r for r in predictions if r["arm"]==key and r["pair_id"]==row["pair_id"])
            outcome=pred["prediction"]
            tint="#aa6117" if outcome=="abstain" else "#21745e" if outcome==pred["actual"] else "#a52b23"
            g.text((25,y+84+j*38),f"{'AB'[j]}: {outcome.replace('_',' ')} ({pred['score']:+.2f})",fill=tint,font_size=17)
        for j,(path,roi) in enumerate((("W_path","W_roi_xyxy"),("inspection_path","inspection_roi_xyxy"))):
            patch=ImageOps.contain(crop_member(archive,row[path],row[roi]),(370,174))
            gallery.paste(patch,(375+380*j,y+14))
gallery.save(run_dir/"development_pairs_with_predictions.jpg",quality=95)
print("Saved comparison and prediction gallery.")
canvas
